In [ ]:
# Accept parameters passed from orchestration notebook via dbutils.notebook.run()
# These simulate DAB variables in the bundle deployment

try:
    # Get parameters from dbutils.widgets (passed by dbutils.notebook.run)
    catalog_name = dbutils.widgets.get("catalog_name")
    schema_prefix = dbutils.widgets.get("schema_prefix")
    print(f"Using parameters from orchestration:")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")
except Exception:
    # Fallback to default values if not called from orchestration
    catalog_name = "dev_catalog"
    schema_prefix = "slv_cdm_hrs"
    print(f"Using default values (not called from orchestration):")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")

In [ ]:
# Dsplay the Databricks Asset Bundles
#==============================================================================
# Dispaly the .../.bundle/hrs_dbx_repo/default/files
# The path is the internal deployment target directory created automatically by Databricks Asset Bundles (DABs).
#
# Use this to determine if all your files were deployed successfully.
#==============================================================================

"""
import os
import sys

print("Current working directory:")
print(os.getcwd())

print("\nChecking bundle files path:")
bundle_path = "/Workspace/Users/peteperez.lv@gmail.com/.bundle/hrs_dbx_repo/default/files"

print("Exists:", os.path.exists(bundle_path))

print("\nFiles in bundle root:")
print(os.listdir(bundle_path))

print("\nPython paths:")
for p in sys.path:
    print(p)
    
"""

In [ ]:
# Create the HRS tables.  Read the SQL file from the Git repository and execute it .
# notes: Create four tables.

from pathlib import Path

# Variables are received from the first cell (either from orchestration or defaults)

# Create an array of SQL files
tables = [
    "create_hrs_cohort.sql",
    #"create_hrs_wave.sql",
    #"create_hrs_respondent.sql",
    #"create_hrs_demographics.sql",
    #"create_hrs_health.sql",
    #"create_hrs_leave_behind.sql",
]

for table in tables:
    print(f"Creating {table}...")
    
    # Read SQL file
    sql_path = Path(f"../../sql/ddl/{table}")
    sql_text = sql_path.read_text()
    
    # Split and execute statements with parameter binding
    statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"  Executing statement {i}/{len(statements)}")
        spark.sql(stmt, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})
    
    print(f"✓ {table} created")